# Aula 1 · Notebook 07 — Sua primeira análise (Bloco 3)

Objetivo: **responder 3 perguntas reais** sobre um dataset de e-commerce brasileiro, e depois inventar a sua quarta pergunta.

Este é o notebook **principal do Bloco 3** da aula.

---

## O dataset: Olist

Dataset público da **Olist**, empresa que conecta pequenos lojistas a marketplaces. ~96 mil pedidos entregues entre 2016 e 2018.

| Coluna | O que é |
|--------|---------|
| `order_id` | ID do pedido |
| `customer_state` | UF do cliente |
| `customer_city` | Cidade do cliente |
| `purchase_date` | Data da compra |
| `price`, `freight_value` | Preço e frete |
| `product_category_en` | Categoria do produto |
| `delivery_days` | Dias até a entrega |
| `delivered_late` | Atrasou? (bool) |
| `review_score` | Estrelas (1-5) |
| `review_comment_message` | Comentário em PT-BR |
| ... (32 colunas no total) |

Tempo estimado: 50 minutos.

## Passo 1 — Baixar o dataset

In [ ]:
import urllib.request
import pathlib

URL = "https://github.com/fenakamuta/poliusppro-data-engineering/releases/download/aula1-olist-2026-v1/olist.parquet"
PATH = pathlib.Path("data/olist.parquet")
PATH.parent.mkdir(exist_ok=True)

if not PATH.exists():
    print("Baixando olist.parquet (~8 MB)...")
    urllib.request.urlretrieve(URL, PATH)
    print(f"✅ Baixado: {PATH.stat().st_size / 1e6:.1f} MB")
else:
    print(f"Já está baixado: {PATH.stat().st_size / 1e6:.1f} MB")

## Passo 2 — Dar uma espiada

In [ ]:
import duckdb

duckdb.sql("SELECT * FROM 'data/olist.parquet' LIMIT 5").df()

In [ ]:
# Quantas linhas e colunas?
duckdb.sql("""
    SELECT COUNT(*) AS pedidos,
           COUNT(DISTINCT customer_state) AS estados,
           COUNT(DISTINCT product_category_en) AS categorias
    FROM 'data/olist.parquet'
""").df()

---

## Pergunta 1 — Qual estado tem o melhor review médio?

**Hipótese:** estados maiores (mais oferta, mais competição) talvez tenham scores melhores.

In [ ]:
duckdb.sql("""
    SELECT customer_state,
           COUNT(*) AS pedidos,
           ROUND(AVG(review_score), 2) AS score_medio
    FROM 'data/olist.parquet'
    GROUP BY customer_state
    ORDER BY score_medio DESC
    LIMIT 10
""").df()

### Discussão

Repare como o `score_medio` quase não varia entre estados — todos próximos de 4. A **média esconde nuance**.

**Para pensar:** se o objetivo é "melhorar a satisfação", olhar a média não ajuda. Precisaríamos olhar a **proporção de notas baixas** por estado.

---

## Pergunta 2 — Quanto a entrega atrasa, e quanto isso impacta?

In [ ]:
duckdb.sql("""
    SELECT delivered_late,
           COUNT(*) AS pedidos,
           ROUND(AVG(delivery_days), 1) AS dias_medio,
           ROUND(AVG(review_score), 2) AS score_medio
    FROM 'data/olist.parquet'
    WHERE delivery_days IS NOT NULL
    GROUP BY delivered_late
""").df()

### Discussão

Pedidos no prazo: ~11 dias, score ~4,3.
Pedidos atrasados: ~34 dias, score ~2,4.

**Atraso vira review ruim.** Esse é o tipo de insight que justifica investir em logística. Vamos voltar nele na Aula 4 quando treinarmos um modelo que prevê risco de atraso.

---

## Pergunta 3 — Quais categorias têm os reviews mais negativos?

Para o time de operações: "onde investigar primeiro".

In [ ]:
duckdb.sql("""
    SELECT product_category_en,
           COUNT(*) AS pedidos,
           ROUND(AVG(review_score), 2) AS score_medio
    FROM 'data/olist.parquet'
    WHERE product_category_en IS NOT NULL
    GROUP BY product_category_en
    HAVING pedidos > 500
    ORDER BY score_medio ASC
    LIMIT 10
""").df()

### Discussão

Categorias com mais reviews ruins são candidatas a investigação: produto com defeito? Embalagem ruim? Expectativa errada na descrição?

**Engenheiro de dados entrega "onde olhar primeiro"; o time de produto investiga.**

---

## Pergunta 4 — Sua vez

Pense numa pergunta que **você teria interesse** em responder. Algumas ideias:

- Qual o **valor médio de frete** por categoria?
- Quem demora mais para entregar: vendedores do **mesmo estado** do cliente, ou de outro estado?
- Existe **mês do ano** em que os reviews são piores?
- Quanto custa em média uma **`bed_bath_table`** comparada a **`computers_accessories`**?

Escreva sua query abaixo (e celebre quando funcionar):

In [ ]:
# Sua query aqui:

duckdb.sql("""
    -- COLE SUA QUERY AQUI
""").df()

---

## Passo final — Salvar e versionar no GitHub

Sua análise está pronta? Vamos salvar e mandar pro GitHub.

In [ ]:
import datetime

resumo = duckdb.sql("""
    SELECT customer_state,
           COUNT(*) AS pedidos,
           ROUND(AVG(review_score), 2) AS score,
           ROUND(AVG(delivery_days), 1) AS dias_medio
    FROM 'data/olist.parquet'
    GROUP BY customer_state
    ORDER BY pedidos DESC
""").df()

caminho = "data/minha_analise_olist.csv"
resumo.to_csv(caminho, index=False)
print(f"✅ Análise salva em {caminho}")
print(f"   Gerada em {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}")

### Comitar no Git

No terminal (Cursor → Ctrl/Cmd + ` para abrir):

```bash
git init                        # se ainda não tiver
git add data/minha_analise_olist.csv 07-bloco-3-olist-analise.ipynb
git commit -m "minha primeira análise de dados real"
```

Para subir no GitHub, crie um repositório vazio na sua conta e:

```bash
git remote add origin https://github.com/SEU_USUARIO/SEU_REPO.git
git branch -M main
git push -u origin main
```

Pronto: você tem um portfólio começando.

---

## Resumo da aula

| Você sabia? | Agora sabe |
|-------------|-----------|
| Que dado vinha em CSV | Que vem em CSV, JSON e Parquet — e cada um serve pra algo |
| Que SQL precisava de servidor | Que DuckDB roda SQL direto em arquivo |
| Que análise era em planilha | Que análise pode ser em código, versionada no Git |

## Tarefa para a Aula 2 (23/06)

Escolha **uma API pública** que te interesse. Sugestões: PokéAPI, Hacker News, OpenWeather, IBGE.

Na semana que vem, vamos coletar dados dessa API **automaticamente, sem código**, usando uma ferramenta chamada **n8n**. Você não vai acreditar.

---

🎉 **Parabéns. Sua primeira análise real está pronta.**